# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [1]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [2]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [4]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[-0.1793,  0.4209,  0.7534,  0.3878, -0.5589, -0.6270, -0.2596, -0.6401],
        [-0.0617, -0.1415, -0.1030, -0.0294, -0.1364,  0.2161,  0.0188, -0.0269]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [5]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cpu
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [6]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [7]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[ 0.6004, -0.8482],
        [-0.5168, -1.6187],
        [-0.4935,  0.0397],
        [ 0.5313, -0.8682],
        [-0.4684, -0.4313]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[-0.1584, -0.2380],
        [-0.5185, -0.3158],
        [-0.2280, -0.3207],
        [-0.1482, -0.3127],
        [-0.3068, -0.3877]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [9]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


Output no_grad method:
 tensor([[ 0.4481, -0.0645],
        [-0.0386,  0.9896],
        [ 0.5926,  0.3269],
        [-0.1077,  0.0069],
        [-0.1500, -0.0095]])
Output inference_mode method:
 tensor([[ 0.4481, -0.0645],
        [-0.0386,  0.9896],
        [ 0.5926,  0.3269],
        [-0.1077,  0.0069],
        [-0.1500, -0.0095]])
Output no_grad context manager:
 tensor([[ 0.4481, -0.0645],
        [-0.0386,  0.9896],
        [ 0.5926,  0.3269],
        [-0.1077,  0.0069],
        [-0.1500, -0.0095]])
Output inference_mode context manager:
 tensor([[ 0.4481, -0.0645],
        [-0.0386,  0.9896],
        [ 0.5926,  0.3269],
        [-0.1077,  0.0069],
        [-0.1500, -0.0095]])


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [10]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc.weight: requires_grad=False
fc.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [11]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


CustomBlock set to train mode
CustomBlock set to eval mode
CustomBlock set to eval mode
Output training mode: tensor([[0.0000, 2.6554, 0.0000, 0.0000],
        [0.0000, 0.0000, 1.4186, 0.0000]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[0.3255, 1.3277, 1.4631, 0.0000],
        [0.0000, 0.0000, 0.7093, 0.6171]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [12]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [13]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [14]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])


## Type hints and Shape Information

For production code, a practical pattern is to use `Tensor` for type hints and document the expected shape next to the signature.

`Tensor` tells us what kind of object is expected, while `[B, C, H, W]` documents the tensor dimensions explicitly.


In [15]:
from torch import Tensor

def normalize_images(x: Tensor) -> Tensor:
    # x: [B, C, H, W]
    return x / 255.0



## Работа на семинаре

### LSTM

![lstm](assets/LSTM.png)

In [16]:
import torch
from torch import nn


class LSTMCell(nn.Module):
    def __init__(self, input_size: int, hidden_size: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.gates = nn.Linear(input_size + hidden_size, 4 * hidden_size)

    def forward(self, x, state=None):
        if state is None:
            h = x.new_zeros(x.shape[0], self.hidden_size)
            c = x.new_zeros(x.shape[0], self.hidden_size)
        else:
            h, c = state
        f, g, i, o = self.gates(torch.cat((x, h), dim=-1)).chunk(4, dim=-1)
        c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        h = torch.sigmoid(o) * torch.tanh(c)
        return h, c


lstm = LSTMCell(4, 8)
x = torch.randn(2, 5, 4)
state = None
outputs = []
for x_t in x.unbind(dim=1):
    state = lstm(x_t, state)
    outputs.append(state[0])
output = torch.stack(outputs, dim=1)
print(output.shape)


torch.Size([2, 5, 8])


### Inception

![inception](assets/inception.png)

In [19]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, channels_1x1, reduce_3x3, channels_3x3,
                 reduce_5x5, channels_5x5, pool_channels):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels, channels_1x1, 1),
                nn.ReLU()
            ),
            nn.Sequential(
                nn.Conv2d(in_channels, reduce_3x3, 1),
                nn.ReLU(),
                nn.Conv2d(reduce_3x3, channels_3x3, 3, padding=1),
                nn.ReLU()
            ),
            nn.Sequential(
                nn.Conv2d(in_channels, reduce_5x5, 1),
                nn.ReLU(),
                nn.Conv2d(reduce_5x5, channels_5x5, 5, padding=2),
                nn.ReLU()
            ),
            nn.Sequential(
                nn.MaxPool2d(3, stride=1, padding=1),
                nn.Conv2d(in_channels, pool_channels, 1),
                nn.ReLU()
            )
        ])

    def forward(self, x):
        return torch.cat([branch(x) for branch in self.branches], dim=1)


inception = InceptionBlock(16, 8, 8, 16, 4, 8, 8)
output = inception(torch.randn(2, 16, 16, 16))
print(output.shape)


torch.Size([2, 40, 16, 16])


### SE

![se](assets/SqueezeAndExcite.png)

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = None,
                 reduction: int = 16, stride: int = 1):
        super().__init__()
        if out_channels is None:
            out_channels = in_channels
        hidden_channels = max(1, out_channels // reduction)
        self.transform = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride,
                      padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Conv2d(out_channels, hidden_channels, 1),
            nn.ReLU(),
            nn.Conv2d(hidden_channels, out_channels, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        u = self.transform(x)
        weights = self.excitation(self.squeeze(u))
        return u * weights


se = SEBlock(in_channels=16, out_channels=32, stride=2)
output = se(torch.randn(2, 16, 16, 16))
print(output.shape)


### Selective Kernel

![selective](assets/SelectiveKernel.png)


In [18]:
class SelectiveKernelBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        self.channels = channels
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size, padding=kernel_size // 2,
                          bias=False),
                nn.BatchNorm2d(channels),
                nn.ReLU()
            )
            for kernel_size in (3, 5)
        ])
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fuse = nn.Sequential(
            nn.Conv2d(channels, max(1, channels // reduction), 1),
            nn.ReLU()
        )
        self.select = nn.Conv2d(max(1, channels // reduction), 2 * channels, 1)

    def forward(self, x):
        features = torch.stack([branch(x) for branch in self.branches], dim=1)
        z = self.fuse(self.pool(features.sum(dim=1)))
        weights = self.select(z).reshape(x.shape[0], 2, self.channels, 1, 1)
        weights = torch.softmax(weights, dim=1)
        return (features * weights).sum(dim=1)


sk = SelectiveKernelBlock(16)
output = sk(torch.randn(2, 16, 16, 16))
print(output.shape)


torch.Size([2, 16, 16, 16])


### PatchMerger

![patchmerger](assets/PatchMerger.png)


In [20]:
class PatchMerger(nn.Module):
    def __init__(self, dim: int, num_output_tokens: int):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.queries = nn.Parameter(torch.randn(num_output_tokens, dim) * 0.02)

    def forward(self, x):
        x = self.norm(x)
        scores = self.queries @ x.transpose(-2, -1)
        weights = torch.softmax(scores, dim=-1)
        return weights @ x


patch_merger = PatchMerger(dim=32, num_output_tokens=4)
output = patch_merger(torch.randn(2, 16, 32))
print(output.shape)


torch.Size([2, 4, 32])


## Домашнее задание

## Задание 1, реализуйте требуемые блоки

## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385

In [21]:
import torch
from torch import nn


class ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.layers(x)

## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357

In [22]:
class SeparableConv2d(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 0
    ):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels
        )
        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)
        return self.pointwise(x)
        
block = SeparableConv2d(16, 32, kernel_size=3, padding=1)
x = torch.randn(2, 16, 32, 32)

output = block(x)
print(output.shape)

torch.Size([2, 32, 32, 32])


## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [23]:

class VanillaAttention(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.align = nn.Linear(dim, dim, bias=False)
        self.value = nn.Linear(dim, dim, bias=False)
        self.query = nn.Linear(dim, dim, bias=False)

    def forward(self, query: torch.Tensor, key: torch.Tensor) -> torch.Tensor:
        scores = torch.bmm(key, self.align(query).unsqueeze(-1)).squeeze(-1)
        attention = torch.softmax(scores, dim=1)
        context = torch.bmm(attention.unsqueeze(1), key).squeeze(1)
        return torch.tanh(self.value(context) + self.query(query))


vanilla_attention = VanillaAttention(dim=8)
query = torch.randn(2, 8)
key = torch.randn(2, 5, 8)
output = vanilla_attention(query, key)
print(output.shape)


torch.Size([2, 8])


## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [24]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, query: torch.Tensor, key: torch.Tensor,
                value: torch.Tensor) -> torch.Tensor:
        scores = (query @ key.transpose(-2, -1)) / (query.shape[-1] ** 0.5)
        attention = torch.softmax(scores, dim=-1)
        return attention @ value


dot_product_attention = ScaledDotProductAttention()
query = torch.randn(2, 3, 8)
key = torch.randn(2, 5, 8)
value = torch.randn(2, 5, 8)
output = dot_product_attention(query, key, value)
print(output.shape)


torch.Size([2, 3, 8])


## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762


In [25]:
class MultiHeadAttention(nn.Module):
    def __init__(self, dim: int, num_heads: int):
        super().__init__()
        if dim <= 0 or num_heads <= 0 or dim % num_heads != 0:
            raise ValueError("dim must be positive and divisible by num_heads")
        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.query_proj = nn.Linear(dim, dim)
        self.key_proj = nn.Linear(dim, dim)
        self.value_proj = nn.Linear(dim, dim)
        self.attention = ScaledDotProductAttention()
        self.output_proj = nn.Linear(dim, dim)

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, length, _ = x.shape
        return x.reshape(batch_size, length, self.num_heads, self.head_dim).transpose(1, 2)

    def forward(self, query: torch.Tensor, key: torch.Tensor,
                value: torch.Tensor) -> torch.Tensor:
        q = self.split_heads(self.query_proj(query))
        k = self.split_heads(self.key_proj(key))
        v = self.split_heads(self.value_proj(value))
        heads = self.attention(q, k, v)
        merged = heads.transpose(1, 2).reshape(query.shape[0], query.shape[1], self.dim)
        return self.output_proj(merged)


multihead_attention = MultiHeadAttention(dim=16, num_heads=4)
query = torch.randn(2, 3, 16)
key = torch.randn(2, 5, 16)
value = torch.randn(2, 5, 16)
output = multihead_attention(query, key, value)
print(output.shape)


torch.Size([2, 3, 16])


## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](assets/TransformerEncoder.png)


https://arxiv.org/abs/1706.03762

In [26]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, dim: int, num_heads: int,
                 hidden_dim: int = None, dropout: float = 0.1):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = 4 * dim
        self.norm1 = nn.LayerNorm(dim)
        self.attention = MultiHeadAttention(dim, num_heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim),
            nn.Dropout(dropout)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, L, dim]; normalization precedes each residual branch.
        normalized = self.norm1(x)
        x = x + self.attention(normalized, normalized, normalized)
        return x + self.mlp(self.norm2(x))


transformer_encoder = TransformerEncoderLayer(dim=16, num_heads=4)
x = torch.randn(2, 5, 16)
output = transformer_encoder(x)
print(output.shape)


torch.Size([2, 5, 16])


## MLP Mixer (0.1 балл)


![MLPMixer](assets/MLPMixer.png)


https://arxiv.org/abs/2105.01601

In [27]:
class MLPMixerBlock(nn.Module):
    def __init__(self, num_patches: int, dim: int,
                 token_hidden_dim: int, channel_hidden_dim: int):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.token_mlp = nn.Sequential(
            nn.Linear(num_patches, token_hidden_dim),
            nn.GELU(),
            nn.Linear(token_hidden_dim, num_patches)
        )
        self.norm2 = nn.LayerNorm(dim)
        self.channel_mlp = nn.Sequential(
            nn.Linear(dim, channel_hidden_dim),
            nn.GELU(),
            nn.Linear(channel_hidden_dim, dim)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, num_patches, dim]
        tokens = self.norm1(x).transpose(1, 2)
        x = x + self.token_mlp(tokens).transpose(1, 2)
        return x + self.channel_mlp(self.norm2(x))


mlp_mixer = MLPMixerBlock(num_patches=16, dim=32,
                          token_hidden_dim=8, channel_hidden_dim=64)
x = torch.randn(2, 16, 32)
output = mlp_mixer(x)
print(output.shape)


torch.Size([2, 16, 32])


## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)


https://arxiv.org/abs/2201.09792

In [28]:
class ConvMixer(nn.Module):
    def __init__(self, channels: int, kernel_size: int = 3):
        super().__init__()
        if kernel_size <= 0 or kernel_size % 2 == 0:
            raise ValueError("kernel_size must be a positive odd integer")
        self.depthwise = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size,
                      padding=kernel_size // 2, groups=channels),
            nn.ReLU(),
            nn.BatchNorm2d(channels)
        )
        self.pointwise = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm2d(channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, channels, H, W]
        x = x + self.depthwise(x)
        return self.pointwise(x)


conv_mixer = ConvMixer(channels=16, kernel_size=5)
x = torch.randn(2, 16, 32, 32)
output = conv_mixer(x)
print(output.shape)


torch.Size([2, 16, 32, 32])


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

Ответ: 

MLP Mixer, ConvMixer и Multihead Attention решают похожую задачу: смешивают информацию между разными участками изображения и преобразуют их признаки. Они могут давать близкое качество, потому что attention — не единственный способ учитывать связи между участками. MLP Mixer делает это через полносвязные слои, а ConvMixer — через свёртки. При этом близкое качество зависит от задачи, данных и обучения, а не гарантируется самой архитектурой.



Общая формула:

$$
Y = X + \operatorname{TokenMixing}(X)
$$

Различие заключается в способе смешивания токенов:

- **Multi-Head Attention** — динамические веса, зависящие от входных данных.
- **ConvMixer** — локальное смешивание через свёртку.
- **MLP-Mixer** — глобальное смешивание через MLP.



| Метод | Преимущества | Недостатки |
|---|---|---|
| **Multihead Attention** | Глобальные связи за один слой; адаптивные веса; поддержка разной длины последовательности | Обычный attention строит матрицу размера N × N, что дорого по памяти и вычислениям |
| **MLPMixer** | Глобальное смешивание позиций; простая архитектура | Token MLP привязан к числу токенов; веса смешивания не адаптируются к входу; плотные слои могут быть дорогими |
| **ConvMixer** | Эффективные свёртки; учитывает локальную структуру изображения; поддерживает разные пространственные размеры | Дальние связи требуют нескольких слоёв или больших ядер; ядра не адаптируются к содержимому входа |
